In [ ]:
!pip uninstall datasets -y datasets fsspec gcsfs
!pip install datasets==2.19.0 fsspec==2024.3.1 gcsfs==2024.3.1 transformers evaluate seqeval accelerate -q

In [ ]:
from datasets import load_dataset
dataset = load_dataset("tner/conll2003")
dataset

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3453
    })
})

In [ ]:
dataset["train"].features

{'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'tags': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None)}

In [ ]:
label_set = set()

for example in dataset["train"]:
    for tag in example["tags"]:
        label_set.add(tag)

pos_labels = sorted(list(label_set))

pos_labels

[0, 1, 2, 3, 4, 5, 6, 7, 8]

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_and_align_labels(example):

    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:

        if word_idx is None:
            labels.append(-100)

        elif word_idx != previous_word_idx:
            labels.append(example["tags"][word_idx])

        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(pos_labels),
    id2label={i:str(i) for i in pos_labels},
    label2id={str(i):i for i in pos_labels}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="bert-token",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.069061,0.056107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,0.069061,0.056107
2,0.032961,0.054141


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3512, training_loss=0.07880371824280276, metrics={'train_runtime': 6916.17, 'train_samples_per_second': 4.06, 'train_steps_per_second': 0.508, 'total_flos': 312165168002100.0, 'train_loss': 0.07880371824280276, 'epoch': 2.0})

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [str(p) for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    true_labels = [
        [str(l) for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return results

In [ ]:
trainer.evaluate()

{'eval_loss': 0.05414115637540817,
 'eval_runtime': 242.0593,
 'eval_samples_per_second': 13.426,
 'eval_steps_per_second': 1.681,
 'epoch': 2.0}

In [ ]:
from transformers import pipeline

nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

nlp("John works at Google in California")

[{'entity_group': '3',
  'score': np.float32(0.9950736),
  'word': 'john',
  'start': 0,
  'end': 4},
 {'entity_group': '0',
  'score': np.float32(0.9989187),
  'word': 'works at',
  'start': 5,
  'end': 13},
 {'entity_group': '1',
  'score': np.float32(0.98216635),
  'word': 'google',
  'start': 14,
  'end': 20},
 {'entity_group': '0',
  'score': np.float32(0.99578875),
  'word': 'in',
  'start': 21,
  'end': 23},
 {'entity_group': '5',
  'score': np.float32(0.99531895),
  'word': 'california',
  'start': 24,
  'end': 34}]


**POS Tagging**
Assigns grammatical labels to words.
Works at word level.
Example: noun, verb, adjective.
Easier task.
Chunking
Groups words into meaningful phrases.
Works at phrase level.
Example: noun phrase, verb phrase.
Medium complexity.

Differences
POS tagging identifies grammar of words, while chunking identifies phrase structure.
Challenges
Subword token alignment
Handling special tokens
Training time
Observations
BERT understands context effectively.
DistilBERT provides faster training with good performance.


**Token Classification using BERT for POS Tagging & Chunking**
**Objective**
The objective of this assignment is to build a token classification model using a transformer architecture (DistilBERT) to perform Part-of-Speech (POS) tagging and chunking tasks. The model assigns labels to each token in a sentence.
Difference Between POS Tagging and Chunking
POS Tagging
Assigns grammatical categories to individual words.
Works at word level.
Example tags: Noun, Verb, Adjective.
Easier task because it focuses only on grammar.
**Chunking**
Groups words into meaningful phrases.
Works at phrase level.
Example: Noun Phrase (NP), Verb Phrase (VP).
Slightly complex because relationships between words must be learned.
Challenges Faced
Aligning labels with subword tokens created by BERT tokenizer.
Handling special tokens like [CLS] and [SEP].
Managing ignored labels using value -100.
Training required computational time.
Observations and Insights
Transformer models understand contextual meaning effectively.
DistilBERT provides faster training compared to BERT.
Token classification models achieve good accuracy for sequence labeling tasks.
Proper preprocessing significantly improves performance.
Pipeline Flow

Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference